# E1b: the CTC and Transducer families next to Whisper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e1b_families_colab.ipynb)

Experiment E1b of [RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md)
compares three families of end-to-end models on the same test sets: Whisper (encoder-decoder),
wav2vec2 XLS-R (CTC) and Parakeet TDT (Transducer). Models, revisions and comparisons are fixed in
[`experiments/E1b/models.toml`](https://github.com/ivannamor/multilingual-asr-lab/blob/main/experiments/E1b/models.toml).

This notebook transcribes the FLEURS test sets (Italian, German, English) and the VoxPopuli Italian
test split with the CTC and Transducer models. On VoxPopuli it also runs Whisper-small and
large-v3-turbo, whose FLEURS runs come from E1.

1. On Google Colab: *Runtime → Change runtime type → T4 GPU*, the GPU of the Whisper runs of E1, so
   that the real-time factors compare. Then *Runtime → Run all* and keep the tab open. The first
   run installs the package and restarts the session: then *Run all* again.
2. At the end the browser downloads `e1b_runs.zip` (if it does not, download it from the *Files*
   panel): the predictions and `run.json` of every transcription, no audio and no weights.
3. In a local clone, from the repository root: `python3 -m zipfile -e e1b_runs.zip .`

If the session is interrupted, run all cells again: a transcription resumes where it stopped.

## 1. Clone and install

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Work inside a clone of the repository, so that every run.json records the git commit and the
# run directories end up where the experiment scripts expect them.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    # The hf extra adds librosa and, if Colab's is too old, a newer Transformers
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[hf]"], check=True)
    # pip may have upgraded packages that the kernel has already imported (numpy), and mixing two
    # versions in one process fails later: restart the session, then run all the cells again
    # (this one will skip the installation)
    print("Installed: the session restarts. Run all the cells again.", flush=True)
    os.kill(os.getpid(), 9)
print(Path.cwd())

## 2. Settings

In [ ]:
SPEC_PATH = Path("experiments/E1b/models.toml")
RUNS_DIR = Path("experiments/runs")
HALLUCINATION_SILENCE = 2.0  # the decoding selected in E2, for the Whisper runs on VoxPopuli

### GPU setup

Same setup as the quickstart notebook: install the CUDA 12 libraries that CTranslate2 is built for
and load them explicitly. Unlike the quickstart, this notebook needs the GPU and stops if it is not
usable.

In [ ]:
import ctypes

import ctranslate2
import numpy as np

if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    check=True,
)

In [ ]:
import nvidia.cublas
import nvidia.cudnn
from faster_whisper import WhisperModel

pending = [
    path
    for package in (nvidia.cublas, nvidia.cudnn)
    for path in sorted(Path(next(iter(package.__path__))).glob("lib/*.so*"))
]
for _ in range(5):  # the libraries depend on each other: retry the ones that failed
    failed = []
    for path in pending:
        try:
            ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            failed.append(path)
    pending = failed
    if not pending:
        break

check = WhisperModel("tiny", device="cuda", compute_type="float16")
list(check.transcribe(np.zeros(16_000, dtype=np.float32), language="it")[0])
print("GPU ready")

## 3. Data

The FLEURS test sets (about 1.5 GB) and the VoxPopuli test split (about 0.9 GB), exactly as the
local scripts download them.

In [ ]:
import tomllib

from asr_lab.data.ingest import ingest_fleurs
from asr_lab.data.voxpopuli import ingest_voxpopuli, load_data_spec

spec = tomllib.loads(SPEC_PATH.read_text(encoding="utf-8"))
for test_set in spec["test_sets"]:
    manifest = Path(test_set["manifest"])
    if not manifest.exists():
        if test_set["name"].startswith("fleurs"):
            ingest_fleurs(test_set["lang"], "test", manifest.parent)
        else:
            ingest_voxpopuli(load_data_spec(Path("experiments/E7/test.toml")))
    print(manifest)

## 4. Transcription

One model at a time, freeing the GPU between them.

In [ ]:
import gc

import torch

from asr_lab.inference.faster_whisper import FasterWhisperModel
from asr_lab.inference.run import transcribe_manifest
from asr_lab.inference.transformers_models import FAMILIES

run_dirs = []


def transcribe(model, test_set: dict, name: str) -> None:
    run_dir = RUNS_DIR / name
    print(f"=== {name}")
    transcribe_manifest(model, Path(test_set["manifest"]), run_dir)
    run_dirs.append(run_dir)


def free(model) -> None:
    del model
    gc.collect()
    torch.cuda.empty_cache()


# The Transducer: one model for every test set
checkpoint = spec["models"]["tdt"]["all"]
model = FAMILIES["tdt"](checkpoint["id"], revision=checkpoint["revision"], device="cuda")
for test_set in spec["test_sets"]:
    transcribe(model, test_set, f"e1b-tdt-{test_set['name']}")
free(model)

# CTC: one checkpoint per language, Italian on two test sets
for lang, checkpoint in spec["models"]["ctc"].items():
    model = FAMILIES["ctc"](checkpoint["id"], revision=checkpoint["revision"], device="cuda")
    for test_set in spec["test_sets"]:
        if test_set["lang"] == lang:
            transcribe(model, test_set, f"e1b-ctc-{test_set['name']}")
    free(model)

# Whisper on VoxPopuli (on FLEURS, its runs come from E1)
voxpopuli = next(t for t in spec["test_sets"] if t["name"] == "voxpopuli-it")
for size, short in (("small", "small"), ("large-v3-turbo", "turbo")):
    model = FasterWhisperModel(
        size, device="cuda", hallucination_silence_threshold=HALLUCINATION_SILENCE
    )
    transcribe(model, voxpopuli, f"e1b-whisper-{short}-voxpopuli-it")
    free(model)

## 5. Download the results

In [ ]:
import json
import zipfile

archive = Path("e1b_runs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for run_dir in run_dirs:
        for file in ("predictions.jsonl", "run.json"):
            zipped.write(run_dir / file, (run_dir / file).as_posix())
        run = json.loads((run_dir / "run.json").read_text())
        print(f"{run_dir.name}: RTF {run['rtf'] or float('nan'):.3f}")
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass